In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
np.random.seed(42)
n = 500
rng = np.random.default_rng(42)

In [ ]:
# --- Base data ---
passengers = np.random.randint(200, 5000, n).astype(object)
delay_min = np.random.choice([np.nan, 0, 5, 10, 15, 30], n)
zone = list(np.random.choice(['North','South','East','West','Central'], n))
route_id = list(np.random.choice([f'R{i:02d}' for i in range(1,51)], n))
dates = list(pd.date_range('2022-01-01', periods=n, freq='D'))

In [ ]:
# --- NOISE INJECTION (do NOT modify below) ---
idx = rng.choice(n, size=40, replace=False)
for i in idx[:15] : 
    passengers[i] = np.nan # missing values
for i in idx[15:25]:
    passengers[i] = str(passengers[i] or 0).split('.')[0] + ' pax' 
for i in idx[25:32]: 
    zone[i] = zone[i].lower() 
for i in idx[32:37]: 
    route_id[i] = ' ' + route_id[i] + ' ' 
for i in idx[37:40]: 
    delay_min[i] = -999 
for i in rng.choice(n, 8): 
    passengers[i] = 99999 

In [ ]:
df = pd.DataFrame({'route_id': route_id, 'date': dates,
 'passengers': passengers, 'delay_min': delay_min, 'zone': zone})

In [ ]:
df.to_csv("dataset")

In [ ]:
df["passengers"] = (df["passengers"].astype(str).str.replace(" pax", "", regex=False))

In [ ]:
df["passengers"] = pd.to_numeric(df["passengers"], errors="coerce")

In [ ]:
median_passengers = df["passengers"].median()
df["passengers"] = df["passengers"].fillna(median_passengers)

In [ ]:
mean = df["passengers"].mean()
std = df["passengers"].std()
upper_limit = mean + 3 * std
p99 = df["passengers"].quantile(0.99)

In [ ]:
df.loc[df["passengers"] > upper_limit, "passengers"] = p99

In [ ]:
#Task 2
df["delay_min"] = df["delay_min"].replace(-999, np.nan)
route_median = df.groupby("route_id")["delay_min"].transform("median")
df["delay_min"] = df["delay_min"].fillna(route_median)

In [ ]:
df.isnull().sum()

In [ ]:
route_median = df.groupby("route_id")["delay_min"].transform("median")
df["delay_min"] = df["delay_min"].fillna(route_median)

In [ ]:
df["delay_min"] = df["delay_min"].fillna(df["delay_min"].median())

In [ ]:
df["zone"] = (df["zone"].astype(str).str.strip().str.title())

In [ ]:
valid_zones = ["North", "South", "East", "West", "Central"]
df.loc[~df["zone"].isin(valid_zones), "zone"] = np.nan

In [ ]:
df["zone"] = df["zone"].fillna(df["zone"].mode()[0])
df["route_id"] = df["route_id"].astype(str).str.strip()

In [ ]:
df["peak_load"] = np.select(
    [df["passengers"] > 3500,(df["passengers"] >= 1500) & (df["passengers"] <= 3500)],
    [ "High","Medium"],
    default="Low"
)

In [ ]:
df["day_of_week"] = df["date"].dt.day_name()

In [ ]:
df.to_csv("cleaned_dataset.csv", index=False)

In [ ]:
route_stats = (df.groupby("route_id").agg(avg_passengers=("passengers", "mean"),avg_delay=("delay_min", "mean")).reset_index())
route_stats

In [ ]:
route_list = [(row.route_id, row.avg_passengers, row.avg_delay)for row in route_stats.itertuples(index=False)]
route_list

In [ ]:
def merge(left, right, key_index, reverse=True):
    merged = []
    i = j = 0

    while i < len(left) and j < len(right):
        if reverse:
            if left[i][key_index] >= right[j][key_index]:
                merged.append(left[i])
                i += 1
            else:
                merged.append(right[j])
                j += 1
        else:
            if left[i][key_index] <= right[j][key_index]:
                merged.append(left[i])
                i += 1
            else:
                merged.append(right[j])
                j += 1

    merged.extend(left[i:])
    merged.extend(right[j:])
    return merged


In [ ]:
def merge_sort(arr, key_index, reverse=True):
    if len(arr) <= 1:
        return arr

    mid = len(arr) // 2

    left = merge_sort(arr[:mid], key_index, reverse)
    right = merge_sort(arr[mid:], key_index, reverse)

    return merge(left, right, key_index, reverse)
def merge_sort(arr, key_index, reverse=True):
    if len(arr) <= 1:
        return arr

    mid = len(arr) // 2

    left = merge_sort(arr[:mid], key_index, reverse)
    right = merge_sort(arr[mid:], key_index, reverse)

    return merge(left, right, key_index, reverse)


In [ ]:
sorted_by_passengers = merge_sort(route_list, key_index=1, reverse=True)

print("\nTop 5 Routes by Average Passengers")
print(f"{'Route':<10}{'Avg Passengers':>18}{'Avg Delay':>15}")

for r in sorted_by_passengers[:5]:
    print(f"{r[0]:<10}{r[1]:>18.2f}{r[2]:>15.2f}")

In [ ]:
efficiency_list = []

for route_id, avg_passengers, avg_delay in route_list:
    efficiency_score = avg_passengers / (1 + avg_delay)
    efficiency_list.append(
        (route_id, avg_passengers, avg_delay, efficiency_score)
    )

# Rank by efficiency score
sorted_efficiency = merge_sort(efficiency_list, key_index=3, reverse=True)
sorted_efficiency

In [ ]:
print("\nTop 5 Routes by Efficiency Score")
print("-" * 70)
print(f"{'Route':<10}{'Passengers':>15}{'Delay':>12}{'Efficiency':>18}")
for r in sorted_efficiency[:5]:
    print(f"{r[0]:<10}{r[1]:>15.2f}{r[2]:>12.2f}{r[3]:>18.2f}")
print("\nBottom 5 Routes by Efficiency Score")
print("-" * 70)
print(f"{'Route':<10}{'Passengers':>15}{'Delay':>12}{'Efficiency':>18}")
for r in sorted_efficiency[-5:]:
    print(f"{r[0]:<10}{r[1]:>15.2f}{r[2]:>12.2f}{r[3]:>18.2f}")


In [ ]:
for i in range(len(sorted_efficiency) - 1):
    assert (
        sorted_efficiency[i][3] >= sorted_efficiency[i + 1][3]
    ), "Merge Sort verification failed!"

print("\n Merge Sort verification passed!")
print("Efficiency scores are in non-increasing order.")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
daily = (df.groupby("date")["passengers"].sum().reset_index())
print("Daily",daily)
rolling = daily["passengers"].rolling(window=7, min_periods=1).mean()
print("rolling:",rolling)

In [ ]:
axes[0, 0].plot(daily["date"],daily["passengers"],label="Daily Passengers",color="steelblue",linewidth=1.5)
axes[0, 0].plot(daily["date"],rolling,label="7-Day Rolling Mean",color="red",linewidth=2)
axes[0, 0].set_title("Daily Total Passengers")
axes[0, 0].set_xlabel("Date")
axes[0, 0].set_ylabel("Passengers")
axes[0, 0].legend()

In [ ]:
grouped = (
    df.groupby(["zone", "peak_load"])["passengers"].mean().unstack(fill_value=0))
print(grouped)
zones = grouped.index
x = np.arange(len(zones))
width = 0.25
peak_levels = ["Low", "Medium", "High"]

for i, level in enumerate(peak_levels):
    values = grouped[level] if level in grouped.columns else np.zeros(len(zones))
    axes[0, 1].bar(
        x + (i - 1) * width,
        values,
        width,
        label=level
    )

axes[0, 1].set_xticks(x)
axes[0, 1].set_xticklabels(zones)
axes[0, 1].set_title("Average Passengers by Zone and Peak Load")
axes[0, 1].set_xlabel("Zone")
axes[0, 1].set_ylabel("Average Passengers")
axes[0, 1].legend()

In [ ]:
delay = df["delay_min"].dropna()
mean = np.mean(delay)
std = np.std(delay)

axes[1, 0].hist(
    delay,
    bins=20,
    density=True,
    alpha=0.7,
    color="skyblue",
    edgecolor="black",
    label="Observed"
)

x = np.linspace(delay.min(), delay.max(), 300)

normal = (
    1 / (std * np.sqrt(2 * np.pi))
) * np.exp(
    -(x - mean) ** 2 / (2 * std ** 2)
)

axes[1, 0].plot(
    x,
    normal,
    color="red",
    linewidth=2,
    label="Normal Curve"
)

axes[1, 0].set_title("Delay Distribution")
axes[1, 0].set_xlabel("Delay (minutes)")
axes[1, 0].set_ylabel("Density")
axes[1, 0].legend()

In [ ]:
route_summary = (
    df.groupby("route_id")
      .agg(
          avg_passengers=("passengers", "mean"),
          avg_delay=("delay_min", "mean"),
          zone=("zone", lambda x: x.mode()[0])
      )
      .reset_index()
)

colors = {
    "North": "blue",
    "South": "green",
    "East": "orange",
    "West": "purple",
    "Central": "red"
}

for zone in colors:
    subset = route_summary[route_summary["zone"] == zone]

    axes[1, 1].scatter(
        subset["avg_passengers"],
        subset["avg_delay"],
        color=colors[zone],
        label=zone,
        alpha=0.75,
        s=60
    )

top10 = route_summary.nlargest(10, "avg_passengers")

for _, row in top10.iterrows():
    axes[1, 1].text(
        row["avg_passengers"],
        row["avg_delay"],
        row["route_id"],
        fontsize=8
    )

axes[1, 1].set_title("Route Performance")
axes[1, 1].set_xlabel("Average Passengers")
axes[1, 1].set_ylabel("Average Delay (minutes)")
axes[1, 1].legend(title="Zone")


In [ ]:
plt.tight_layout()
plt.savefig("mobility_dashboard.png",dpi=150,bbox_inches="tight")
plt.show()

In [ ]:
import folium
from folium.plugins import HeatMap

In [ ]:
ZONE_COORDS = {
    'North': [28.8000, 77.2090],
    'South': [28.4595, 77.0266],
    'East': [28.6139, 77.3910],
    'West': [28.6139, 77.0270],
    'Central': [28.6139, 77.2090],
}

In [ ]:

city_map = folium.Map(location=[28.6139, 77.2090],zoom_start=11)

In [ ]:
zone_stats = (
    df.groupby("zone")
      .agg(
          avg_passengers=("passengers", "mean"),
          avg_delay=("delay_min", "mean"),
          dominant_peak=("peak_load", lambda x: x.mode()[0])
      )
)
marker_count = 0

In [ ]:
for zone, row in zone_stats.iterrows():

    lat, lon = ZONE_COORDS[zone]

    avg_passengers = row["avg_passengers"]
    avg_delay = row["avg_delay"]
    dominant_peak = row["dominant_peak"]

    # Marker color
    if avg_passengers < 2000:
        color = "green"
    elif avg_passengers <= 3500:
        color = "orange"
    else:
        color = "red"

    radius = avg_passengers / 200

    popup = folium.Popup(
        f"""
        <b>Zone:</b> {zone}<br>
        <b>Avg Passengers:</b> {avg_passengers:.2f}<br>
        <b>Avg Delay:</b> {avg_delay:.2f} min<br>
        <b>Dominant Peak Load:</b> {dominant_peak}
        """,
        max_width=250
    )

    folium.CircleMarker(
        location=[lat, lon],
        radius=radius,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.75,
        popup=popup
    ).add_to(city_map)

    marker_count += 1

In [ ]:
heat_data = []

rng = np.random.default_rng(42)

for _, row in df.iterrows():

    lat, lon = ZONE_COORDS[row["zone"]]

    # Random jitter ±0.02 degrees
    lat += rng.uniform(-0.02, 0.02)
    lon += rng.uniform(-0.02, 0.02)

    heat_data.append([lat, lon, row["passengers"]])

HeatMap(
    heat_data,
    radius=15,
    blur=10,
    max_zoom=13
).add_to(city_map)

In [ ]:
city_map.save("city_mobility_map.html")

print(f"Map saved as 'city_mobility_map.html'")
print(f"Total markers added: {marker_count}")